
# Multi‑Reference, Multi‑Run Text Generation Evaluation

This notebook evaluates **6 model variants** with **3 stochastic runs per model** across **5 prompt sets**, using **3 expert references** per prompt.
It computes **BERTScore (semantic)**, **SacreBLEU**, and **chrF++** with both **max-over-refs** and **mean-over-refs** variants, and aggregates by **runs**, **prompt sets**, and overall (**macro/micro averages**). It also provides **stratified paired bootstrap** testing between models.

> Files expected in your working directory (edit `FILE_PATHS` below as needed):
>
> - `predictions_real3.xlsx`
> - `predictions_real4.xlsx`
> - `predictions_real5.xlsx`
> - `predictions_real6.xlsx`
>
> If you have more/other files, add them to `FILE_PATHS`.


In [ ]:

# %% [markdown]
# ## 0. Install dependencies
# Prefer the `calamine` engine (no openpyxl). Falls back to openpyxl if needed.
# NOTE: Comment out installs if your env already has them.

%pip -q install pandas numpy sacrebleu bert-score pandas-calamine regex


In [ ]:

# %%
import re
import math
import json
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from IPython.display import display

import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

from bert_score import score as bertscore_score

# ------------------ User Config ------------------
# Add/modify your Excel files here:
FILE_PATHS = [
    "predictions_real3.xlsx",
    "predictions_real4.xlsx",
    "predictions_real5.xlsx",
    "predictions_real6.xlsx",
]

# Optionally choose a language/model for BERTScore.
# If LANG == "auto", a heuristic will pick 'zh' if many CJK chars are present, else 'en'.
LANG = "auto"  # "auto" | "en" | "zh" | any valid bert-score lang code
BERT_MODEL_EN = "roberta-large"         # for English
BERT_MODEL_ZH = "bert-base-chinese"     # for Chinese
BERT_USE_IDF  = True
BERT_BASELINE = True  # rescale_with_baseline (recommended)

# chrF++ settings (standard-ish)
CHRF_CHAR_ORDER  = 6
CHRF_WORD_ORDER  = 2  # word n-grams → chrF++
CHRF_BETA        = 2

# Bootstrap settings
N_BOOT = 1000
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# If your column names don't match the auto-detection,
# you can enforce them via OVERRIDES below.
OVERRIDES = dict(
    # set_col="set",                       # e.g., "set" / "prompt_set" / "batch"
    # prompt_id_col="prompt_id",           # unique prompt id within a set
    # reference_cols=["ref1","ref2","ref3"],
    # model_run_regex=r"^(?P<model>.+?)_r(?P<run>\d+)$",  # parse model/run from column names
)


In [ ]:

# %%
def read_excel_any(path: str | Path) -> pd.DataFrame:
    '''
    Try pandas-calamine first (no openpyxl), then fall back to openpyxl.
    Reads the first sheet. Edit if you need a specific sheet name.
    '''
    import pandas as pd
    path = str(path)
    try:
        return pd.read_excel(path, engine="calamine")
    except Exception:
        try:
            return pd.read_excel(path, engine="openpyxl")
        except Exception as e:
            raise RuntimeError(f"Cannot read {path}. Install pandas-calamine or openpyxl. Error: {e}")

def load_all_files(paths) -> pd.DataFrame:
    frames = []
    for p in paths:
        df = read_excel_any(p)
        df["__source_file__"] = Path(p).name
        frames.append(df)
    # Align columns (outer join) then concat
    all_cols = sorted(set().union(*[set(f.columns) for f in frames]))
    frames = [f.reindex(columns=all_cols) for f in frames]
    return pd.concat(frames, ignore_index=True)

df_raw = load_all_files(FILE_PATHS)
print("Loaded shape:", df_raw.shape)
display(df_raw.head(3))


In [ ]:

# %%
META_CANDIDATES = ["set","prompt_set","batch","split","subset","topic","domain"]
PROMPT_ID_CANDS = ["prompt_id","id","qid","example_id","uid","index"]
PROMPT_TEXT_CANDS = ["prompt","instruction","question","input","source"]

def detect_columns(df: pd.DataFrame):
    cols = list(df.columns)
    lower = {c: c.lower() for c in cols}
    # Detect set column
    set_col = None
    for c in cols:
        if lower[c] in META_CANDIDATES:
            set_col = c
            break
    # Detect prompt id
    prompt_id_col = None
    for c in cols:
        if lower[c] in PROMPT_ID_CANDS:
            prompt_id_col = c
            break
    # Detect prompt text
    prompt_text_col = None
    for c in cols:
        if lower[c] in PROMPT_TEXT_CANDS:
            prompt_text_col = c
            break

    # Reference columns (any of: ref, reference, gold) up to 3
    ref_cols = []
    for c in cols:
        lc = lower[c]
        if re.search(r"\b(ref|reference|gold|answer)\b", lc):
            ref_cols.append(c)
    # If too many, take first 3 by natural sort
    if len(ref_cols) > 3:
        ref_cols = sorted(ref_cols, key=lambda x: (re.sub(r'\D','',x) or '0', x))[:3]

    # Candidate model/run columns = non-meta, non-ref
    exclude = set(ref_cols + [set_col, prompt_id_col, prompt_text_col, "__source_file__"])
    model_run_cols = [c for c in cols if c not in exclude and not c.startswith("__")]
    return dict(
        set_col=set_col,
        prompt_id_col=prompt_id_col,
        prompt_text_col=prompt_text_col,
        reference_cols=ref_cols,
        model_run_cols=model_run_cols,
    )

det = detect_columns(df_raw)
# Apply overrides if provided
for k,v in OVERRIDES.items():
    if v is not None:
        det[k] = v

print("=== Auto-detected mapping ===")
print(json.dumps(det, indent=2))


In [ ]:

# %%
# Try several common patterns
PATTERNS = [
    r"^(?P<model>.+?)_r(?P<run>\d+)$",          # e.g., modelA_r1
    r"^(?P<model>.+?)_run(?P<run>\d+)$",        # e.g., modelA_run2
    r"^(?P<model>.+?)\.seed(?P<run>\d+)$",      # e.g., modelA.seed3
    r"^(?P<model>.+?)\s*\(run\s*(?P<run>\d+)\)$",
]

def parse_model_run(colname: str):
    for pat in [OVERRIDES.get("model_run_regex")] + PATTERNS:
        if not pat:
            continue
        m = re.match(pat, colname)
        if m:
            gd = m.groupdict()
            return gd["model"], int(gd["run"])
    # Fallback: treat whole colname as a model with run=1
    return colname, 1

def build_tidy(df: pd.DataFrame, mapping: dict) -> pd.DataFrame:
    '''
    Build a long/tidy frame: one row per (set, prompt_id, model, run, output, refs)
    '''
    import numpy as np
    set_col        = mapping.get("set_col")
    prompt_id_col  = mapping.get("prompt_id_col")
    prompt_text_col= mapping.get("prompt_text_col")
    ref_cols       = mapping.get("reference_cols", [])
    model_cols     = mapping.get("model_run_cols", [])

    # if set_col missing, use source file as set proxy
    if set_col is None:
        set_col = "__source_file__"

    # if prompt_id missing, create stable index within set
    if prompt_id_col is None:
        df = df.copy()
        df["__row_index__"] = np.arange(len(df))
        prompt_id_col = "__row_index__"

    # Collect rows
    rows = []
    for _, row in df.iterrows():
        set_id = row.get(set_col)
        prompt_id = row.get(prompt_id_col)
        prompt_txt = row.get(prompt_text_col) if prompt_text_col in df.columns else None
        refs = [str(row[c]) for c in ref_cols if pd.notna(row.get(c))]
        for c in model_cols:
            out = row.get(c)
            if pd.isna(out):
                continue
            model, run = parse_model_run(c)
            rows.append(dict(
                set_id=set_id,
                prompt_id=prompt_id,
                prompt_text=prompt_txt,
                model=model,
                run=run,
                output=str(out),
                references=refs,
            ))
    tidy = pd.DataFrame(rows)
    return tidy

tidy = build_tidy(df_raw, det)
print("Tidy shape:", tidy.shape)
display(tidy.head(5))
print("Models detected:", sorted(tidy['model'].unique()))
print("Runs detected:", sorted(tidy['run'].unique()))
print("Avg #refs:", tidy['references'].map(len).mean())


In [ ]:

# %%
CJK_RANGES = [
    (0x4E00, 0x9FFF),   # CJK Unified Ideographs
    (0x3400, 0x4DBF),   # CJK Unified Ideographs Extension A
    (0x20000, 0x2A6DF), # Extension B
    (0x2A700, 0x2B73F), # Extension C
    (0x2B740, 0x2B81F), # Extension D
    (0x2B820, 0x2CEAF), # Extension E
]

def is_cjk_char(ch):
    cp = ord(ch)
    return any(lo <= cp <= hi for lo,hi in CJK_RANGES)

def cjk_ratio(text: str) -> float:
    if not isinstance(text, str) or not text:
        return 0.0
    cjk = sum(1 for ch in text if is_cjk_char(ch))
    return cjk / max(1, len(text))

def auto_pick_lang(samples: list[str]) -> str:
    '''
    If many CJK chars, choose 'zh', else 'en'.
    '''
    import numpy as np
    ratios = [cjk_ratio(s) for s in samples if isinstance(s, str)]
    if len(ratios) == 0:
        return "en"
    return "zh" if np.mean(ratios) >= 0.15 else "en"

# Decide BERTScore lang & model
if LANG == "auto":
    probe = []
    probe.extend([x for x in tidy["output"].head(200)])
    for refs in tidy["references"].head(200):
        probe.extend(refs)
    pick = auto_pick_lang(probe)
    print("Auto-picked language:", pick)
    if pick == "zh":
        BERT_LANG = "zh"
        BERT_MODEL = BERT_MODEL_ZH
    else:
        BERT_LANG = "en"
        BERT_MODEL = BERT_MODEL_EN
else:
    BERT_LANG = LANG
    BERT_MODEL = BERT_MODEL_ZH if LANG.startswith("zh") else BERT_MODEL_EN

print("BERTScore language:", BERT_LANG)
print("BERTScore model   :", BERT_MODEL)


In [ ]:

# %%
bleu_metric = BLEU(effective_order=True)  # sacrebleu with brevity penalty
chrf_metric = CHRF(char_order=CHRF_CHAR_ORDER, word_order=CHRF_WORD_ORDER, beta=CHRF_BETA)

def bleu_sentence(hyp: str, refs: list[str]) -> dict:
    # Per-ref scores
    per_ref = [sacrebleu.sentence_bleu(hyp, [r]).score for r in refs] if refs else [0.0]
    return dict(mean=float(np.mean(per_ref)), max=float(np.max(per_ref)))

def chrf_sentence(hyp: str, refs: list[str]) -> dict:
    if not refs:
        return dict(mean=0.0, max=0.0)
    per_ref = [chrf_metric.sentence_score(hyp, [r]).score for r in refs]
    return dict(mean=float(np.mean(per_ref)), max=float(np.max(per_ref)))

def bertscore_batch_max(cands: list[str], refs_list: list[list[str]]) -> np.ndarray:
    '''
    BERTScore with multiple references using the library's multi-ref support.
    Returns F1 per cand by taking max over refs.
    '''
    P, R, F1 = bertscore_score(
        cands, refs_list,
        lang=BERT_LANG, model_type=BERT_MODEL,
        idf=BERT_USE_IDF, rescale_with_baseline=BERT_BASELINE,
        verbose=False
    )
    return F1.detach().cpu().numpy()

def bertscore_batch_mean(cands: list[str], refs_list: list[list[str]]) -> np.ndarray:
    '''
    Compute mean over references by scoring against each reference set separately and averaging.
    '''
    # Transpose refs_list into per-ref lists
    max_refs = max(len(r) for r in refs_list) if refs_list else 0
    if max_refs <= 1:
        # Single-ref case: mean == max
        return bertscore_batch_max(cands, refs_list)
    # Normalize lengths (pad with empty string to keep alignment)
    per_ref_arrays = []
    for ref_idx in range(max_refs):
        one_refs = [ (rs[ref_idx] if ref_idx < len(rs) else "") for rs in refs_list ]
        P, R, F1 = bertscore_score(
            cands, one_refs,
            lang=BERT_LANG, model_type=BERT_MODEL,
            idf=BERT_USE_IDF, rescale_with_baseline=BERT_BASELINE,
            verbose=False
        )
        per_ref_arrays.append(F1.detach().cpu().numpy())
    return np.mean(np.stack(per_ref_arrays, axis=0), axis=0)

def compute_sentence_metrics(df_sub: pd.DataFrame) -> pd.DataFrame:
    '''
    Compute per-(set_id, prompt_id, model, run) metrics:
      - BLEU_mean/max_over_refs
      - chrF_mean/max_over_refs
      - BERTScore_F1_mean/max_over_refs
    '''
    df = df_sub.copy()
    # BLEU/chrF per row
    bleu_mean, bleu_max = [], []
    chrf_mean, chrf_max = [], []
    for hyp, refs in zip(df["output"].tolist(), df["references"].tolist()):
        b = bleu_sentence(hyp, refs)
        c = chrf_sentence(hyp, refs)
        bleu_mean.append(b["mean"]); bleu_max.append(b["max"])
        chrf_mean.append(c["mean"]); chrf_max.append(c["max"])
    df["BLEU_mean"] = bleu_mean
    df["BLEU_max"]  = bleu_max
    df["chrF_mean"] = chrf_mean
    df["chrF_max"]  = chrf_max

    # BERTScore in batch
    cands = df["output"].tolist()
    refs_list = df["references"].tolist()
    bs_max  = bertscore_batch_max(cands, refs_list)
    bs_mean = bertscore_batch_mean(cands, refs_list)
    df["BERT_F1_max"]  = bs_max * 100.0  # to 0-100 scale for comparability
    df["BERT_F1_mean"] = bs_mean * 100.0
    return df

metrics_by_run = compute_sentence_metrics(tidy)
display(metrics_by_run.head(3))


In [ ]:

# %%
METRIC_COLS = ["BLEU_mean","BLEU_max","chrF_mean","chrF_max","BERT_F1_mean","BERT_F1_max"]

# 1) Aggregate across runs per (set, prompt, model)
agg_prompt = (metrics_by_run
              .groupby(["set_id","prompt_id","model"], as_index=False)
              .agg(**{
                  **{f"{m}_meanOverRuns": (m, "mean") for m in METRIC_COLS},
                  **{f"{m}_sdOverRuns"  : (m, "std")  for m in METRIC_COLS},
                  "n_runs": ("run", "nunique")
              }))

# 2) Aggregate within set (mean over prompts)
within_set = (agg_prompt
              .groupby(["set_id","model"], as_index=False)
              .agg(**{f"{m}_setMean": (f"{m}_meanOverRuns","mean") for m in METRIC_COLS}))

# 3) Macro/micro averages across sets
# Macro: equal weight per set
macro = (within_set
         .groupby("model", as_index=False)
         .agg(**{f"{m}_macro": (f"{m}_setMean","mean") for m in METRIC_COLS}))

# Micro: weight by number of prompts per set
prompts_per_set = (agg_prompt.groupby(["set_id","model"])["set_id"]
                   .count().reset_index(name="n_prompts"))
within_set_w = within_set.merge(prompts_per_set, on=["set_id","model"], how="left")

micro = (within_set_w
         .groupby("model", as_index=False)
         .apply(lambda g: pd.Series({
             **{f"{m}_micro": float(np.average(g[f"{m}_setMean"], weights=g["n_prompts"])) for m in METRIC_COLS},
             "n_prompts_total": int(g["n_prompts"].sum())
         }))
        ).reset_index(drop=True)

leaderboard = macro.merge(micro, on="model", how="left")

# Also compute robustness across sets: CV and min/max set performance (using BERT_F1_max by default)
def robustness(df_within_set: pd.DataFrame, metric="BERT_F1_max_setMean"):
    rows = []
    for model, g in df_within_set.groupby("model"):
        vals = g[metric].dropna().values
        if len(vals) == 0:
            cv = np.nan; vmin=np.nan; vmax=np.nan
        else:
            vmin = float(np.min(vals)); vmax=float(np.max(vals))
            mu = float(np.mean(vals)); sd = float(np.std(vals))
            cv = sd / mu if mu != 0 else np.nan
        rows.append(dict(model=model, min_set=vmin, max_set=vmax, cv=cv))
    return pd.DataFrame(rows)

rob = robustness(within_set, metric="BERT_F1_max_setMean")

leaderboard = leaderboard.merge(rob, on="model", how="left")

# Sort by primary semantic metric (macro average of BERT_F1_max)
leaderboard = leaderboard.sort_values(by="BERT_F1_max_macro", ascending=False)
display(leaderboard.round(3))


In [ ]:

# %%
def paired_bootstrap_stratified(agg_runs_df: pd.DataFrame, modelA: str, modelB: str,
                                metric="BERT_F1_max", n_boot=1000, seed=42):
    '''
    Stratified paired bootstrap over prompt sets.
    For each bootstrap:
      - For each set: sample prompts with replacement
      - For each sampled prompt: pick a run uniformly for A and B
      - Compute set means then macro-average across sets
    Returns CI for A-B.
    '''
    rng = np.random.default_rng(seed)
    # Build index: dict[(set_id, prompt_id, model)] -> list of per-run metric values
    key2runs = defaultdict(list)
    for _, row in agg_runs_df.iterrows():
        k = (row["set_id"], row["prompt_id"], row["model"])
        key2runs[k].append(row[metric])

    # Collect sets and prompts
    sets = sorted(agg_runs_df["set_id"].dropna().unique().tolist())
    set2prompts = {s: sorted(agg_runs_df.loc[agg_runs_df["set_id"]==s, "prompt_id"].unique().tolist())
                   for s in sets}

    diffs = []
    for _ in range(n_boot):
        set_means_A = []
        set_means_B = []
        for s in sets:
            prompts = set2prompts[s]
            if not prompts:
                continue
            sampled_prompts = rng.choice(prompts, size=len(prompts), replace=True)
            vals_A, vals_B = [], []
            for p in sampled_prompts:
                runs_A = key2runs.get((s,p,modelA), [])
                runs_B = key2runs.get((s,p,modelB), [])
                if runs_A:
                    vals_A.append(rng.choice(runs_A))
                if runs_B:
                    vals_B.append(rng.choice(runs_B))
            if vals_A:
                set_means_A.append(np.mean(vals_A))
            if vals_B:
                set_means_B.append(np.mean(vals_B))
        if set_means_A and set_means_B:
            diffs.append(np.mean(set_means_A) - np.mean(set_means_B))
    diffs = np.array(diffs)
    if len(diffs) == 0:
        return dict(ci_low=np.nan, ci_high=np.nan, mean_diff=np.nan)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return dict(ci_low=float(ci_low), ci_high=float(ci_high), mean_diff=float(np.mean(diffs)))

# Example: compare top-2 by BERT_F1_max_macro
if len(leaderboard) >= 2:
    top2 = leaderboard["model"].head(2).tolist()
    mA, mB = top2[0], top2[1]
    print(f"Bootstrap test: {mA} (A) vs {mB} (B) on BERT_F1_max (macro via stratified sampling)")
    result = paired_bootstrap_stratified(metrics_by_run, mA, mB,
                                         metric="BERT_F1_max", n_boot=N_BOOT, seed=RANDOM_SEED)
    print(json.dumps(result, indent=2))


In [ ]:

# %%
OUTDIR = Path("eval_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

metrics_by_run.to_csv(OUTDIR/"per_run_metrics.csv", index=False)
# Recompute agg_prompt here for saving scope
METRIC_COLS = ["BLEU_mean","BLEU_max","chrF_mean","chrF_max","BERT_F1_mean","BERT_F1_max"]
agg_prompt = (metrics_by_run
              .groupby(["set_id","prompt_id","model"], as_index=False)
              .agg(**{
                  **{f"{m}_meanOverRuns": (m, "mean") for m in METRIC_COLS},
                  **{f"{m}_sdOverRuns"  : (m, "std")  for m in METRIC_COLS},
                  "n_runs": ("run", "nunique")
              }))
within_set = (agg_prompt
              .groupby(["set_id","model"], as_index=False)
              .agg(**{f"{m}_setMean": (f"{m}_meanOverRuns","mean") for m in METRIC_COLS}))

leaderboard.to_csv(OUTDIR/"leaderboard.csv", index=False)
agg_prompt.to_csv(OUTDIR/"per_prompt_agg.csv", index=False)
within_set.to_csv(OUTDIR/"per_set_agg.csv", index=False)

print("Saved:")
for p in ["per_run_metrics.csv","per_prompt_agg.csv","per_set_agg.csv","leaderboard.csv"]:
    print(" -", OUTDIR/p)



## Notes & Tips

- **Column mapping**: If auto-detection picks the wrong columns for references or sets, set `OVERRIDES` at the top.
- **BERTScore**:
  - Primary ranking uses `BERT_F1_max` (semantic, tolerant of paraphrases).
  - `BERT_F1_mean` reflects consistency across the 3 references.
- **BLEU/chrF++**:
  - Report both mean-over-refs and max-over-refs variants as complements.
- **Macro vs Micro**:
  - **Macro** = equal weight per set (fairness across prompt sets).
  - **Micro** = weight by prompts per set (overall average).
- **Significance**:
  - Use the stratified paired bootstrap to compare any two models (`paired_bootstrap_stratified`).
